In [0]:
from pyspark.sql.functions import col, substring

In [0]:
# 1. Lendo da Bronze
df_bronze_ibge = spark.table("oftalmo_sus.01_bronze.bronze_dim_municipios_ibge")
nome_tabela_silver = "oftalmo_sus.02_silver.tb_dim_municipios_ibge"

In [0]:
# 2. Transformação Silver
df_silver_ibge = (
    df_bronze_ibge
    .select(
        # Cortar o 7º dígito para bater com o DATASUS
        substring(col("codigo_ibge").cast("string"), 1, 6).alias("codigo_ibge_6"),
        col("nome").alias("nome_municipio"),
        col("latitude"),
        col("longitude")
    )
    # Qualidade de Dados: Remover se não tiver coordenada ou código
    .dropna(subset=["codigo_ibge_6", "latitude", "longitude"])
    # Garantir que a tabela dimensão não tenha códigos duplicados
    .dropDuplicates(["codigo_ibge_6"]) 
)

print(f"💾 Salvando Dimensão refinada em: {nome_tabela_silver}...")

In [0]:
# 3. Carga Delta na Silver
(
    df_silver_ibge.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(nome_tabela_silver)
)

print("✅ Dimensão Geográfica perfeitamente alinhada com o SUS e salva na Silver!")